<font size="6" color='grey'> <b>

Generative KI. Verstehen. Anwenden. Gestalten.
</b></font> </br>

<font size="5" color='grey'> <b>
M09 - Aufgabe A2: YOLO Object Detection → Koordinaten-Maske → Comic
</b></font> </br>

---

# Aufgabenbeschreibung

Verbesserter Workflow mit **YOLO (You Only Look Once)** für robuste Objekterkennung:

1. **YOLO Object Detection**: Schnelle und präzise Objekterkennung mit exakten Koordinaten
2. **Koordinaten-basierte Maske**: Erstelle Maske innerhalb der erkannten Koordinaten
3. **Comic-Transformation**: Wende Comic-Effekt auf die Maske an

**Vorteil gegenüber A1**: YOLO liefert zuverlässigere Ergebnisse und exakte Bounding-Box-Koordinaten.

---

# 1 | Umgebung einrichten

In [ ]:
#@title 🔧 Umgebung einrichten { display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/GenAI.git#subdirectory=04_modul
from genai_lib.utilities import check_environment, get_ipinfo, setup_api_keys, mprint, install_packages
setup_api_keys(['OPENAI_API_KEY', 'HF_TOKEN'], create_globals=False)
print()
check_environment()
print()
get_ipinfo()

In [ ]:
#@title 🛠️ Installationen { display-mode: "form" }
install_packages([
    'ultralytics>=8.0.0',
    'opencv-python>=4.8.0',
    'scikit-image>=0.21.0',
    'pillow>=10.0.0'
])

In [ ]:
#@title 📂 Testbilder herunterladen { display-mode: "form" }
!rm -rf files
!mkdir -p files

# Diverse Test-Bilder mit klaren Objekten
!curl -L https://raw.githubusercontent.com/ralf-42/GenAI/main/02_daten/02_bild/peoples.png -o files/peoples.png
!curl -L https://raw.githubusercontent.com/ralf-42/GenAI/main/02_daten/02_bild/apfel.png -o files/apfel.png

print("✅ Testbilder heruntergeladen")

# 2 | Imports & Setup

In [ ]:
import cv2
import numpy as np
from ultralytics import YOLO
from PIL import Image as PILImage
from IPython.display import display, Image as IPImage, Markdown
import matplotlib.pyplot as plt
import os

print("✅ Imports erfolgreich")

# 3 | Schritt 1: YOLO Object Detection

## 3.1 | YOLO Modell laden

In [ ]:
# YOLO Modell laden (YOLOv8 Medium)
print("📥 Lade YOLO v8 Modell (erster Start kann länger dauern)...")
model = YOLO('yolov8m.pt')
print("✅ YOLO Modell geladen")

## 3.2 | Objekterkennung durchführen

In [ ]:
def detect_objects_yolo(image_path, conf_threshold=0.5):
    """
    Erkennt Objekte mit YOLO.
    Gibt Bild, Erkennungen und Koordinaten zurück.
    """
    # Bild laden
    image = cv2.imread(image_path)
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    h, w = image_rgb.shape[:2]
    
    # YOLO Vorhersage
    results = model.predict(source=image_path, conf=conf_threshold, verbose=False)
    
    # Erkannte Objekte extrahieren
    detections = []
    if results[0].boxes is not None:
        for box in results[0].boxes:
            # Bounding Box Koordinaten (x1, y1, x2, y2)
            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().astype(int)
            conf = box.conf.cpu().numpy()[0]
            cls = int(box.cls.cpu().numpy()[0])
            class_name = model.names[cls]
            
            detections.append({
                'class_name': class_name,
                'confidence': conf,
                'bbox': (x1, y1, x2, y2),
                'center': ((x1 + x2) // 2, (y1 + y2) // 2),
                'area': (x2 - x1) * (y2 - y1)
            })
    
    return image_rgb, detections, results

# Test mit peoples.png
test_image_path = "files/peoples.png"
image_rgb, detections, yolo_results = detect_objects_yolo(test_image_path, conf_threshold=0.5)

print(f"✅ Bild geladen: {image_rgb.shape}")
print(f"✅ {len(detections)} Objekte erkannt")

# Erkannte Objekte anzeigen
print("\n📋 Erkannte Objekte:")
for i, det in enumerate(sorted(detections, key=lambda x: x['confidence'], reverse=True), 1):
    print(f"  {i}. {det['class_name']:15} | Confidence: {det['confidence']:.2f} | Größe: {det['area']} px²")

## 3.3 | Bounding Boxes visualisieren

In [ ]:
def draw_bboxes(image_rgb, detections):
    """
    Zeichnet Bounding Boxes auf das Bild.
    """
    image_with_boxes = image_rgb.copy()
    
    # Farben für verschiedene Klassen
    colors = {
        'person': (0, 255, 0),
        'apple': (255, 0, 0),
        'orange': (255, 165, 0)
    }
    
    for det in detections:
        x1, y1, x2, y2 = det['bbox']
        class_name = det['class_name']
        conf = det['confidence']
        
        # Farbe wählen
        color = colors.get(class_name, (0, 255, 255))  # Cyan als Default
        
        # Box zeichnen
        cv2.rectangle(image_with_boxes, (x1, y1), (x2, y2), color, 2)
        
        # Label mit Confidence
        label = f"{class_name} {conf:.2f}"
        cv2.putText(image_with_boxes, label, (x1, y1 - 10),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)
        
        # Center-Punkt
        cx, cy = det['center']
        cv2.circle(image_with_boxes, (cx, cy), 3, color, -1)
    
    return image_with_boxes

image_with_boxes = draw_bboxes(image_rgb, detections)

# Visualisierung
plt.figure(figsize=(12, 8))
plt.imshow(image_with_boxes)
plt.title(f"YOLO Object Detection - {len(detections)} Objekte erkannt")
plt.axis('off')
plt.tight_layout()
os.makedirs('output', exist_ok=True)
plt.savefig('output/01_yolo_detection.png', dpi=100, bbox_inches='tight')
plt.show()

print("✅ Detection visualisiert")

# 4 | Schritt 2: Koordinaten-basierte Maske

## 4.1 | Maske von Bounding Box erstellen

In [ ]:
def create_mask_from_bbox(image_shape, bbox, expand_percentage=0):
    """
    Erstellt eine Maske basierend auf Bounding Box Koordinaten.
    
    Args:
        image_shape: (height, width) des Bildes
        bbox: (x1, y1, x2, y2) Koordinaten
        expand_percentage: Prozentsatz zum Expandieren der Box
    
    Returns:
        mask: Binärbild
    """
    h, w = image_shape[:2]
    mask = np.zeros((h, w), dtype=np.uint8)
    
    x1, y1, x2, y2 = bbox
    
    # Optional: Box expandieren
    if expand_percentage > 0:
        box_w = x2 - x1
        box_h = y2 - y1
        expand_x = int(box_w * expand_percentage / 100)
        expand_y = int(box_h * expand_percentage / 100)
        
        x1 = max(0, x1 - expand_x)
        y1 = max(0, y1 - expand_y)
        x2 = min(w, x2 + expand_x)
        y2 = min(h, y2 + expand_y)
    
    # Maske setzen
    mask[y1:y2, x1:x2] = 255
    
    # Morphologische Operationen für glatte Kanten
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    
    # Glätten mit Gaussian Blur
    mask = cv2.GaussianBlur(mask, (5, 5), 0)
    
    return mask

# Beste Detection (höchste Confidence) wählen
best_detection = max(detections, key=lambda x: x['confidence'])
print(f"\n🎯 Beste Detection: {best_detection['class_name']} (Confidence: {best_detection['confidence']:.2f})")
print(f"   Koordinaten: {best_detection['bbox']}")

# Maske erstellen
mask = create_mask_from_bbox(image_rgb.shape, best_detection['bbox'], expand_percentage=5)
print("✅ Maske erstellt")

## 4.2 | Maske und Objekt-Region visualisieren

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Original mit Bounding Box
axes[0].imshow(image_with_boxes)
axes[0].set_title("YOLO Detection mit Bounding Box")
axes[0].axis('off')

# Maske
axes[1].imshow(mask, cmap='gray')
axes[1].set_title("Koordinaten-Maske")
axes[1].axis('off')

# Objekt extrahiert
x1, y1, x2, y2 = best_detection['bbox']
object_region = image_rgb[y1:y2, x1:x2]
axes[2].imshow(object_region)
axes[2].set_title(f"Extrahiertes Objekt: {best_detection['class_name']}")
axes[2].axis('off')

plt.tight_layout()
plt.savefig('output/02_mask_creation.png', dpi=100, bbox_inches='tight')
plt.show()

print("✅ Maske visualisiert")

# 5 | Schritt 3: Comic-Effekt

## 5.1 | Comic-Filter Implementierung

In [ ]:
def apply_comic_effect(image_rgb, mask=None, edge_threshold=50, num_colors=5):
    """
    Wendet einen Comic-Effekt auf das Bild an.
    Nutzt die Maske um nur diese Region zu bearbeiten.
    """
    image_bgr = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2BGR)
    
    # 1. Kantenerkennung (Canny Edge Detection)
    gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(gray, edge_threshold, edge_threshold * 2)
    edges = cv2.medianBlur(edges, 5)
    
    # 2. Bilateral Filter für Glättung (Cartoon-Effekt)
    smoothed = cv2.bilateralFilter(image_bgr, 9, 75, 75)
    
    # 3. Farbquantisierung (K-Means Clustering)
    z = smoothed.reshape((-1, 3))
    z = np.float32(z)
    criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 10, 1.0)
    ret, labels, center = cv2.kmeans(z, num_colors, None, criteria, 10, cv2.KMEANS_RANDOM_CENTERS)
    center = np.uint8(center)
    res = center[labels.flatten()]
    posterized = res.reshape(smoothed.shape)
    
    # 4. Kanten über Bild legen
    edges_bgr = cv2.cvtColor(edges, cv2.COLOR_GRAY2BGR)
    edges_bgr = (edges_bgr > 0).astype(np.uint8) * 255
    comic = cv2.bitwise_and(posterized, cv2.bitwise_not(edges_bgr))
    comic = np.where(edges_bgr > 0, 0, comic)
    
    # Zurück zu RGB
    comic_rgb = cv2.cvtColor(comic, cv2.COLOR_BGR2RGB)
    
    # Maske anwenden - NUR auf die erkannte Region anwenden
    if mask is not None:
        mask_normalized = (mask / 255.0).astype(np.float32)
        mask_3d = np.stack([mask_normalized, mask_normalized, mask_normalized], axis=2)
        comic_rgb = (comic_rgb * mask_3d + image_rgb * (1 - mask_3d)).astype(np.uint8)
    
    return comic_rgb, edges

# Comic-Effekt anwenden
comic_result, edges = apply_comic_effect(image_rgb, mask, edge_threshold=50, num_colors=5)
print("✅ Comic-Effekt angewendet")

## 5.2 | Comic-Effekt Visualisierung

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Original
axes[0, 0].imshow(image_rgb)
axes[0, 0].set_title("Original Image")
axes[0, 0].axis('off')

# Kanten
axes[0, 1].imshow(edges, cmap='gray')
axes[0, 1].set_title("Edge Detection (Canny)")
axes[0, 1].axis('off')

# Maske
axes[1, 0].imshow(mask, cmap='gray')
axes[1, 0].set_title("Maske (in Koordinaten)")
axes[1, 0].axis('off')

# Comic-Effekt
axes[1, 1].imshow(comic_result)
axes[1, 1].set_title("Comic Effect Angewendet")
axes[1, 1].axis('off')

plt.suptitle("Comic-Effekt Pipeline", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('output/03_comic_effect.png', dpi=100, bbox_inches='tight')
plt.show()

print("✅ Comic-Effekt Visualisierung gespeichert")

# 6 | Vollständiger Workflow

## 6.1 | Kompletter Workflow als Funktion

In [ ]:
def yolo_comic_workflow(image_path, output_dir='output', object_index=0, conf_threshold=0.5):
    """
    Vollständiger Workflow:
    1. YOLO Object Detection (mit Koordinaten)
    2. Koordinaten-basierte Maske erstellen
    3. Comic-Effekt anwenden
    """
    os.makedirs(output_dir, exist_ok=True)
    
    # 1. YOLO Detection
    print("🔍 Schritt 1: YOLO Object Detection...")
    image_rgb, detections, _ = detect_objects_yolo(image_path, conf_threshold)
    
    if not detections:
        print("❌ Keine Objekte erkannt!")
        return None
    
    image_with_boxes = draw_bboxes(image_rgb, detections)
    best_detection = max(detections, key=lambda x: x['confidence'])
    print(f"  ✅ {len(detections)} Objekte erkannt")
    print(f"  ✅ Bestes Objekt: {best_detection['class_name']} (Conf: {best_detection['confidence']:.2f})")
    
    # 2. Maske erstellen
    print("📍 Schritt 2: Koordinaten-Maske erstellen...")
    mask = create_mask_from_bbox(image_rgb.shape, best_detection['bbox'], expand_percentage=5)
    print(f"  ✅ Maske erstellt für Bereich: {best_detection['bbox']}")
    
    # 3. Comic Effect
    print("🎨 Schritt 3: Comic-Effekt anwenden...")
    comic_result, edges = apply_comic_effect(image_rgb, mask)
    print(f"  ✅ Comic-Effekt angewendet")
    
    # Speichern
    cv2.imwrite(f'{output_dir}/01_yolo_detection.png', cv2.cvtColor(image_with_boxes, cv2.COLOR_RGB2BGR))
    cv2.imwrite(f'{output_dir}/02_mask.png', mask)
    cv2.imwrite(f'{output_dir}/03_comic_effect.png', cv2.cvtColor(comic_result, cv2.COLOR_RGB2BGR))
    
    print(f"\n✅ Workflow abgeschlossen! Ergebnisse in '{output_dir}/'")
    
    return {
        'original': image_rgb,
        'detected': image_with_boxes,
        'mask': mask,
        'comic': comic_result,
        'detections': detections,
        'best_detection': best_detection
    }

# Workflow ausführen
results = yolo_comic_workflow('files/peoples.png', 'output')

## 6.2 | Finale 4-Panel Übersicht

In [ ]:
# 4-Panel Visualisierung des gesamten Workflows
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Panel 1: Original
axes[0, 0].imshow(results['original'])
axes[0, 0].set_title('1. Original Image', fontsize=14, fontweight='bold')
axes[0, 0].axis('off')

# Panel 2: YOLO Detection
axes[0, 1].imshow(results['detected'])
axes[0, 1].set_title(f'2. YOLO Detection\n(Best: {results["best_detection"]["class_name"]})', 
                      fontsize=14, fontweight='bold')
axes[0, 1].axis('off')

# Panel 3: Coordinates-based Mask
axes[1, 0].imshow(results['mask'], cmap='gray')
axes[1, 0].set_title('3. Koordinaten-Maske\n(von Bounding Box)', fontsize=14, fontweight='bold')
axes[1, 0].axis('off')

# Panel 4: Comic Effect
axes[1, 1].imshow(results['comic'])
axes[1, 1].set_title('4. Comic Effect\n(auf Maske angewendet)', fontsize=14, fontweight='bold')
axes[1, 1].axis('off')

plt.suptitle('YOLO Workflow: Detection → Koordinaten → Maske → Comic', 
             fontsize=16, fontweight='bold', y=0.98)
plt.tight_layout()
plt.savefig('output/workflow_complete.png', dpi=100, bbox_inches='tight')
plt.show()

print("✅ Workflow Complete!")

# 7 | Bonus: Mit mehreren Bildern testen

In [ ]:
# Mit apfel.png testen
print("\n" + "="*60)
print("Test mit apfel.png")
print("="*60 + "\n")

results_apple = yolo_comic_workflow('files/apfel.png', 'output/apfel')

if results_apple:
    # Übersicht
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    axes[0].imshow(results_apple['detected'])
    axes[0].set_title("YOLO Detection")
    axes[0].axis('off')
    
    axes[1].imshow(results_apple['comic'])
    axes[1].set_title("Comic Effect")
    axes[1].axis('off')
    
    plt.tight_layout()
    plt.show()

# 8 | Zusammenfassung & Fazit

## 🎯 Was haben wir gelernt?

### ✅ YOLO Object Detection
- **Schnell & zuverlässig**: Echtzeit-Objekterkennung
- **Exakte Koordinaten**: (x1, y1, x2, y2) in Pixel
- **Confidence Scores**: Bewertung der Erkennungssicherheit
- **Viele Klassen**: Person, Auto, Tier, Obst, etc.

### ✅ Koordinaten-basierte Masken
- **Von Bounding Box zu Maske**: Umwandlung von Koordinaten in Pixel-Masken
- **Morphologische Operationen**: Glatte Kanten mit Close/Open
- **Gaussian Blur**: Weiches Blending mit Original
- **Flexible Region**: Expandierbar, anpassbar

### ✅ Comic-Effekt Pipeline
- **Canny Edge Detection**: Kantenerkennung mit Schwellenwert
- **Bilateral Filter**: Glättung ohne Kantenverwischung
- **K-Means Clustering**: Intelligente Farbquantisierung
- **Maske-Integration**: Effekt nur auf erkannte Region

### ✅ Workflow-Design
- **Modulare Funktionen**: Wiederverwendbar, testbar
- **End-to-End Pipeline**: Von Input bis Output
- **Visualisierung auf allen Stufen**: Debugging leicht
- **Skalierbar**: Mehrere Objekte, verschiedene Bilder

---

## 🚀 Nächste Schritte

1. **Verschiedene Comic-Stile**
   - Sketch Mode (nur Kanten)
   - Watercolor Mode
   - Oil Painting Mode

2. **Video-Verarbeitung**
   - Frame-by-Frame Verarbeitung
   - Echtzeit-Webcam-Support

3. **Mehrere Objekte**
   - Alle erkannten Objekte verarbeiten
   - Verschiedene Effekte pro Klasse

4. **Advanced Styling**
   - Style Transfer mit KI
   - Generative Effects mit genai_lib